# 🛒 Notebook 1 : Léa apprend à converser et à se souvenir

**Formation « Comprendre et développer des agents IA spécialisés »**, module 1.

## L'histoire

**MégaShop** est une boutique en ligne qui vend les essentiels du quotidien : smartphones, écouteurs, baskets, vêtements, sacs...
Elle veut une conseillère virtuelle, **Léa**, capable de discuter avec les clients comme une vraie vendeuse.

Une bonne vendeuse **se souvient** de ses clients. Si Sofia dit qu'elle chausse du 42 et qu'elle adore le blanc, elle ne veut pas le répéter à chaque message, ni la semaine suivante.

Dans ce notebook, tu vas découvrir **comment fonctionne un LLM**, puis donner à Léa **une mémoire**.

## 🎯 Ce que tu vas apprendre

| Étape | Thème | Python pur | LangChain / LangGraph |
|---|---|:---:|:---:|
| 0 | Configuration | ✅ | |
| 1 | Un LLM lit des **tokens**, pas des mots | ✅ | |
| 2 | Il **prédit** le token suivant (et la température) | ✅ | |
| 3 | Parler à un LLM : les rôles et le prompt système | ✅ | ✅ LangChain |
| 4 | Le problème : un LLM **n'a pas de mémoire** | ✅ | |
| 5 | Donner une mémoire : l'historique | ✅ | ✅ LangGraph |
| 6 | La mémoire coûte cher : fenêtre glissante et résumé | ✅ | |
| 7 | La mémoire long terme : un profil client avec **Pydantic** | ✅ | ✅ LangChain |

**Principe** : à chaque étape, on comprend d'abord **en Python pur**, puis on voit **la même chose avec un framework**.

▶️ Exécute les cellules **dans l'ordre**, avec `Maj + Entrée`. Tout est dans ce notebook.

---
## Étape 0/7 · Configuration  `▱▱▱▱▱▱▱`

🎯 **Objectif** : se connecter à OpenAI.

👉 Il faut une clé API dans un fichier `.env` à la racine du dossier, avec cette ligne : `OPENAI_API_KEY=sk-...`

On utilise ici **`gpt-4.1-mini`** : il est rapide, et il permet de régler la **température** et de voir les **probabilités** des mots, deux choses qu'on va explorer.

In [15]:
import os, math
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                   # lit la clé OPENAI_API_KEY dans le fichier .env
assert os.getenv("OPENAI_API_KEY"), "❌ Ajoute OPENAI_API_KEY=sk-... dans le fichier .env, puis redémarre le noyau"

client = OpenAI()
MODELE = "gpt-4.1-mini"
print(f"✅ Connecté à OpenAI : {MODELE}")

✅ Connecté à OpenAI : gpt-4.1-mini


---
## Étape 1/7 · Un LLM lit des tokens, pas des mots  `▰▱▱▱▱▱▱`

🎯 **Objectif** : voir comment un LLM « lit » un texte.

Avant de lire ton texte, le LLM le découpe en petits morceaux appelés **tokens** : un mot entier, un bout de mot, un espace, un symbole...
C'est important pour deux raisons :
- 💶 **on paie au token**, à l'envoi comme à la réception ;
- 📏 un LLM ne peut lire qu'un **nombre limité** de tokens à la fois.

In [16]:
import tiktoken
encodeur = tiktoken.get_encoding("o200k_base")       # le découpage des modèles GPT-4o, 4.1 et 5

texte = "Les Nike Air Force 1 blanches sont à 119,99 € chez MégaShop."
tokens = encodeur.encode(texte)

print(f"{len(texte)} caractères → {len(tokens)} tokens\n")
print(" | ".join(encodeur.decode([t]) for t in tokens))

60 caractères → 20 tokens

Les |  Nike |  Air |  Force |   | 1 |  bl | anches |  sont |  à |   | 119 | , | 99 |  € |  chez |  Mé | ga | Shop | .


👀 Les mots courants forment **un seul token**, alors que les mots rares ou les noms de marque sont **coupés en morceaux**.

Comparons quelques phrases :

In [17]:
phrases = ["Bonjour, je cherche des baskets.",
           "Hello, I am looking for sneakers.",
           "anticonstitutionnellement",
           "😍👟🔥"]
for phrase in phrases:
    print(f"{len(encodeur.encode(phrase)):>3} tokens │ {phrase}")

  7 tokens │ Bonjour, je cherche des baskets.
  8 tokens │ Hello, I am looking for sneakers.
  5 tokens │ anticonstitutionnellement
  4 tokens │ 😍👟🔥


✅ **À retenir**
- Un LLM lit des **tokens**. En français, un token vaut environ ¾ de mot.
- Le français coûte un peu plus de tokens que l'anglais, et les emojis sont coûteux.
- Les tokens sont l'**unité de coût** et de **limite** d'un LLM.

---
## Étape 2/7 · Il prédit le token suivant  `▰▰▱▱▱▱▱`

🎯 **Objectif** : comprendre comment un LLM « écrit ».

Un LLM ne « réfléchit » pas comme nous : il calcule **quel token a le plus de chances de venir ensuite**, l'ajoute, puis recommence, token après token.

On peut demander à OpenAI de nous montrer ces **probabilités** :

In [18]:
reponse = client.chat.completions.create(
    model=MODELE,
    messages=[{"role": "system", "content": "Continue la phrase. Écris uniquement le mot suivant."},
              {"role": "user", "content": "Pour aller courir, j'achète une paire de"}],
    max_tokens=1, logprobs=True, top_logprobs=5, temperature=0,
)
for candidat in reponse.choices[0].logprobs.content[0].top_logprobs:
    proba = math.exp(candidat.logprob)                # logprob -> probabilité entre 0 et 1
    print(f"{candidat.token!r:<14} {'█' * round(proba * 40):<40} {proba:.0%}")

'cha'          ███████████████████████████              68%
'b'            █████████████                            32%
' chaussures'                                           0%
'basket'                                                0%
'chaus'                                                 0%


👀 Le LLM hésite entre `'cha'` (le début de **cha**ussures) et `'b'` (le début de **b**askets). Il ne choisit pas un mot, il choisit un **token**.

### 🌡️ La température : prudent ou créatif ?

La **température** règle la façon de choisir parmi ces candidats :
- **0** : toujours le plus probable. C'est stable et répétable, idéal pour une réponse de SAV.
- **1 ou plus** : on tire au sort, et les candidats moins probables ont leur chance. C'est créatif et varié, idéal pour du marketing.

Demandons 3 slogans publicitaires avec chaque réglage :

In [19]:
consigne = "Écris un slogan de 8 mots maximum pour le hoodie Adidas vendu par MégaShop."

for temperature in (0, 1.3):
    print(f"\n🌡️ Température {temperature}")
    for _ in range(3):
        r = client.chat.completions.create(model=MODELE, temperature=temperature,
                                           messages=[{"role": "user", "content": consigne}])
        print("   •", r.choices[0].message.content)


🌡️ Température 0
   • Adidas chez MégaShop : style et confort garantis !
   • Adidas chez MégaShop : style et confort garantis !
   • Adidas chez MégaShop : style et confort garantis !

🌡️ Température 1.3
   • Confort Adidas, style unique — disponible chez MégaShop !
   • Adidas chez MégaShop : style, confort, performance garantie.
   • MégaShop : Ton hoodie Adidas, style et confort assurés.


✅ **À retenir**
- Un LLM écrit **token par token**, en choisissant parmi les plus probables.
- **Température basse** = réponses stables (SAV, données). **Température haute** = réponses variées (slogans, idées).
- 💡 Les modèles qui « raisonnent » (`gpt-5`, `gpt-5-mini`...) fixent eux-mêmes ce réglage : on ne peut pas le changer.

---
## Étape 3/7 · Parler à un LLM : les rôles  `▰▰▰▱▱▱▱`

🎯 **Objectif** : donner une personnalité à Léa.

On envoie au LLM une **liste de messages**, chacun avec un **rôle** :

| Rôle | Qui parle ? | Exemple |
|---|---|---|
| `system` | Toi, le développeur : la « fiche de poste » | « Tu es Léa, conseillère de MégaShop... » |
| `user` | Le client | « Vous pouvez m'aider à choisir un cadeau ? » |
| `assistant` | Le LLM, c'est-à-dire Léa | « Avec plaisir ! Pour qui est-ce ? » |

Le **prompt système** est invisible pour le client, mais il guide toutes les réponses.

In [20]:
SYSTEME = """Tu es Léa, conseillère de vente de MégaShop, boutique en ligne d'articles du quotidien
(smartphones, écouteurs, baskets, vêtements, sacs). Tu vouvoies toujours le client.
Réponds en français, en 3 phrases maximum, avec un ton chaleureux."""

messages = [{"role": "system", "content": SYSTEME},
            {"role": "user", "content": "Bonjour ! Vous pouvez m'aider à choisir un cadeau ?"}]

reponse = client.chat.completions.create(model=MODELE, messages=messages)
print("💬 Léa :", reponse.choices[0].message.content)
print(f"\n📊 {reponse.usage.prompt_tokens} tokens envoyés, {reponse.usage.completion_tokens} tokens reçus")

💬 Léa : Bonjour ! Avec plaisir, quel type d'article ou quel budget avez-vous en tête pour ce cadeau ? Je suis là pour vous guider afin de trouver le présent parfait.

📊 83 tokens envoyés, 33 tokens reçus


### 🔁 La même chose avec LangChain

LangChain propose une interface **identique pour tous les fournisseurs** (OpenAI, Anthropic, Mistral, Ollama...). Pour changer de fournisseur, tu changes une seule ligne.

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODELE)
reponse = llm.invoke([("system", SYSTEME), ("user", "Bonjour ! Vous pouvez m'aider à choisir un cadeau ?")])
print("💬 Léa :", reponse.content)

💬 Léa : Bonjour ! Avec plaisir, pourriez-vous me dire pour qui est ce cadeau et quel budget vous souhaitez y consacrer ? Je vous aiderai à trouver l'article parfait chez MégaShop.


✅ **À retenir**
- On parle à un LLM avec une **liste de messages** : `system`, `user`, `assistant`.
- Le **prompt système** définit la personnalité et les règles de Léa.

---
## Étape 4/7 · Le problème : un LLM n'a pas de mémoire  `▰▰▰▰▱▱▱`

🎯 **Objectif** : constater que chaque appel repart de zéro.

Sofia se présente, puis pose une deuxième question **dans un appel séparé** :

In [22]:
def demander(question):
    """Un appel isolé : on n'envoie QUE le prompt système et la question."""
    r = client.chat.completions.create(model=MODELE, messages=[{"role": "system", "content": SYSTEME},
                                                              {"role": "user", "content": question}])
    return r.choices[0].message.content

for question in ["Bonjour, je m'appelle Sofia et je chausse du 42.", "Vous vous souvenez de ma pointure ?"]:
    print(f"👤 Sofia : {question}\n💬 Léa   : {demander(question)}\n")

👤 Sofia : Bonjour, je m'appelle Sofia et je chausse du 42.
💬 Léa   : Bonjour Sofia, ravie de vous aider ! Nous avons une belle sélection de baskets en taille 42, souhaitez-vous que je vous en recommande quelques-unes ? N’hésitez pas à me dire votre style préféré.

👤 Sofia : Vous vous souvenez de ma pointure ?
💬 Léa   : Bonjour ! Pourriez-vous me rappeler votre pointure, s’il vous plaît ? Je serai ravie de vous aider à trouver la paire parfaite.



🤔 **Léa a déjà oublié !** Un LLM est **sans état** : il ne garde rien d'un appel à l'autre. C'est comme une vendeuse frappée d'amnésie toutes les 10 secondes.

✅ **À retenir** : la mémoire n'est **pas dans le LLM**. C'est **ton code** qui doit la gérer.

---
## Étape 5/7 · Donner une mémoire : l'historique  `▰▰▰▰▰▱▱`

🎯 **Objectif** : faire en sorte que Léa se souvienne de la conversation.

L'astuce est simple : **à chaque message, on renvoie toute la conversation**.

```
Appel 1 : [système] [Sofia : je chausse du 42]
Appel 2 : [système] [Sofia : je chausse du 42] [Léa : ...] [Sofia : ma pointure ?]
```

### En Python pur : une liste qu'on remplit

In [23]:
historique = [{"role": "system", "content": SYSTEME}]

def discuter(message_client):
    historique.append({"role": "user", "content": message_client})          # 1. on ajoute le message
    r = client.chat.completions.create(model=MODELE, messages=historique)    # 2. on envoie TOUT l'historique
    reponse = r.choices[0].message.content
    historique.append({"role": "assistant", "content": reponse})             # 3. on garde la réponse
    print(f"👤 {message_client}\n💬 {reponse}\n   📊 {r.usage.prompt_tokens} tokens envoyés\n")

discuter("Bonjour, je m'appelle Sofia et je chausse du 42.")
discuter("Vous vous souvenez de ma pointure ?")

👤 Bonjour, je m'appelle Sofia et je chausse du 42.
💬 Bonjour Sofia, ravie de vous aider ! Nous avons plusieurs modèles en taille 42, notamment des baskets très confortables. Souhaitez-vous que je vous recommande une paire en particulier ?
   📊 86 tokens envoyés

👤 Vous vous souvenez de ma pointure ?
💬 Oui, bien sûr, Sofia, vous chaussez du 42 ! N’hésitez pas à me demander si vous cherchez un modèle ou une couleur spécifique. Je suis là pour vous aider à trouver la paire parfaite.
   📊 141 tokens envoyés



🎉 Léa se souvient. Remarque le compteur 📊 : **il augmente**, car on renvoie tout à chaque fois. On y revient à l'étape 6.

### 🔁 La même chose avec LangGraph : une mémoire par client

Une vraie boutique a **des centaines de clients en même temps**. Il faut un historique **par client**.

LangGraph gère ça avec un **checkpointer** : il range chaque conversation dans un « dossier » identifié par un `thread_id`.

In [24]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver

def noeud_lea(etat):
    """Léa lit l'historique de la conversation et répond."""
    return {"messages": [llm.invoke([("system", SYSTEME)] + etat["messages"])]}

graphe = StateGraph(MessagesState)          # l'état = la liste des messages
graphe.add_node("lea", noeud_lea)
graphe.add_edge(START, "lea")
lea = graphe.compile(checkpointer=InMemorySaver())     # le checkpointer = la mémoire, rangée par client

In [25]:
def parler(client_id, message):
    config = {"configurable": {"thread_id": client_id}}       # le « dossier » de ce client
    etat = lea.invoke({"messages": [("user", message)]}, config)
    print(f"👤 {client_id} : {message}\n💬 Léa : {etat['messages'][-1].content}\n")

parler("sofia", "Bonjour, je m'appelle Sofia et je chausse du 42.")
parler("marc", "Salut, moi c'est Marc, je fais du 44.")
parler("sofia", "Vous vous souvenez de ma pointure ?")

👤 sofia : Bonjour, je m'appelle Sofia et je chausse du 42.
💬 Léa : Bonjour Sofia, ravie de faire votre connaissance ! Nous avons plusieurs baskets en taille 42, puis-je vous aider à trouver un modèle qui vous plairait ? N’hésitez pas à me dire quel style ou marque vous préférez.

👤 marc : Salut, moi c'est Marc, je fais du 44.
💬 Léa : Bonjour Marc, ravie de vous aider ! Pour des baskets en taille 44, avez-vous une marque ou un style préféré ? N’hésitez pas à me dire ce que vous recherchez !

👤 sofia : Vous vous souvenez de ma pointure ?
💬 Léa : Bonjour Sofia, oui tout à fait, vous chaussez du 42 ! Souhaitez-vous que je vous propose des modèles de baskets ou plutôt des chaussures d'un autre type ? Je suis là pour vous aider à trouver votre bonheur.



👀 Léa ne mélange pas Sofia et Marc : chaque `thread_id` a **sa propre mémoire**.

✅ **À retenir**
- La mémoire d'une conversation, c'est **l'historique qu'on renvoie** à chaque appel.
- Avec LangGraph, le **checkpointer** stocke un historique **par conversation** (`thread_id`). En production, on le branche sur une base de données (SQLite, Postgres...).

---
## Étape 6/7 · La mémoire coûte cher  `▰▰▰▰▰▰▱`

🎯 **Objectif** : voir les limites de l'historique, et deux solutions.

Continuons la conversation avec Sofia, et observons le compteur 📊 :

In [26]:
discuter("Je cherche des baskets blanches pour aller avec un jean.")
discuter("Mon budget est de 130 € maximum.")
discuter("Et pour ma sœur, une idée de cadeau à moins de 50 € ? Elle adore le sport.")

👤 Je cherche des baskets blanches pour aller avec un jean.
💬 Parfait, Sofia ! Nous avons de jolies baskets blanches en taille 42, idéales pour un look casual avec un jean. Souhaitez-vous que je vous envoie quelques modèles à découvrir ?
   📊 204 tokens envoyés

👤 Mon budget est de 130 € maximum.
💬 Merci pour cette précision, Sofia ! Je vais vous sélectionner des baskets blanches en taille 42 qui correspondent à votre budget de 130 € maximum. Je reviens vers vous rapidement avec de belles propositions.
   📊 263 tokens envoyés

👤 Et pour ma sœur, une idée de cadeau à moins de 50 € ? Elle adore le sport.
💬 Pour votre sœur sportive, Sofia, je vous recommande nos écouteurs sans fil ou une gourde isotherme, parfaits et à moins de 50 €. Ces articles allient praticité et style pour ses séances de sport. Voulez-vous que je vous envoie les options disponibles ?
   📊 333 tokens envoyés



📈 **À chaque message, on renvoie et on repaie toute la conversation.** Avec 50 messages, chaque nouvelle question coûte 50 fois plus cher que la première.
Et même si les modèles récents acceptent des centaines de milliers de tokens, un historique trop long **ralentit** les réponses et **noie** les informations importantes.

### Solution 1 : la fenêtre glissante ✂️

On ne garde que les **N derniers messages**. C'est simple et le coût est fixe... mais Léa **oublie le début** :

In [27]:
def fenetre_glissante(messages, n=4):
    """Garde le prompt système + les n derniers messages."""
    return [messages[0]] + messages[1:][-n:]

courte = fenetre_glissante(historique) + [{"role": "user", "content": "Rappelez-moi mon prénom et ma pointure ?"}]
r = client.chat.completions.create(model=MODELE, messages=courte)
print("✂️ Fenêtre glissante :", r.choices[0].message.content)

✂️ Fenêtre glissante : Bien sûr, Sofia ! Vous recherchez des baskets blanches en taille 42. N’hésitez pas si vous avez d’autres questions !


### Solution 2 : le résumé 📝

On demande au LLM de **résumer** les anciens messages en quelques lignes, puis on remplace l'historique par ce résumé et les derniers messages. Léa garde l'essentiel, pour beaucoup moins de tokens :

In [29]:
def resumer(messages):
    """Demande au LLM un résumé court de la conversation, centré sur le client."""
    texte = "\n".join(f"{m['role']} : {m['content']}" for m in messages[1:])
    r = client.chat.completions.create(model=MODELE, messages=[{"role": "user", "content":
        "Résume cette conversation de vente en 3 lignes. Garde les infos sur le client "
        "(prénom, tailles, goûts, budget) :\n\n" + texte}])
    return r.choices[0].message.content

resume = resumer(historique)
print("📝 Résumé :", resume)

📝 Résumé : Sofia chausse du 42 et cherche des baskets blanches confortables pour aller avec un jean, avec un budget maximum de 130 €. Elle souhaite également un cadeau sportif pour sa sœur à moins de 50 €. L’assistant propose plusieurs modèles de baskets adaptés et des idées de cadeaux pratiques comme des écouteurs sans fil ou une gourde isotherme.


In [30]:
compacte = [historique[0], {"role": "system", "content": f"Résumé de la conversation : {resume}"},
            *historique[-2:], {"role": "user", "content": "Rappelez-moi mon prénom et ma pointure ?"}]
r = client.chat.completions.create(model=MODELE, messages=compacte)
print("📝 Avec le résumé :", r.choices[0].message.content)
complet = sum(len(encodeur.encode(m["content"])) for m in historique)
print(f"   📊 {r.usage.prompt_tokens} tokens envoyés. L'historique complet en contient déjà {complet}, et il grandit à chaque message.")

📝 Avec le résumé : Bien sûr, Sofia, vous chaussez du 42. Puis-je vous aider à choisir parmi nos baskets blanches confortables ?
   📊 251 tokens envoyés. L'historique complet en contient déjà 348, et il grandit à chaque message.


✅ **À retenir**

| Stratégie | ✅ Avantage | ❌ Inconvénient |
|---|---|---|
| Historique complet | Rien n'est perdu | Coût et lenteur qui augmentent |
| Fenêtre glissante | Simple, coût fixe | Oublie le début |
| Résumé | Garde l'essentiel, peu de tokens | Un appel en plus, des détails perdus |

En pratique, on **combine** souvent : un résumé des anciens messages + les derniers messages en entier.

---
## Étape 7/7 · La mémoire long terme : un profil client  `▰▰▰▰▰▰▰`

🎯 **Objectif** : que Léa se souvienne de Sofia **la semaine prochaine**, dans une nouvelle conversation.

L'historique et le résumé sont du **texte libre**. Pour une mémoire durable, il vaut mieux une **fiche client structurée**, comme dans un logiciel de gestion client (CRM) : prénom, pointure, couleurs préférées, budget...

On va demander au LLM de **remplir cette fiche** à partir de la conversation. Pour garantir le format, on utilise **Pydantic**, la bibliothèque Python de référence pour décrire et **valider** des données.

In [32]:
from pydantic import BaseModel, Field

class ProfilClient(BaseModel):
    """La fiche client que Léa garde en mémoire d'une visite à l'autre."""
    prenom: str | None = Field(description="Prénom du client")
    pointure: int | None = Field(description="Pointure de chaussures")
    couleurs_preferees: list[str] = Field(description="Couleurs que le client aime")
    budget_max: float | None = Field(description="Budget maximum en euros")
    centres_interet: list[str] = Field(description="Sports, loisirs, personnes à qui offrir...")

### En Python pur : les « sorties structurées » d'OpenAI

`client.chat.completions.parse()` oblige le LLM à répondre **exactement** au format de la classe. Pydantic vérifie ensuite les types : la pointure est bien un **nombre entier**, les couleurs une **liste**...

In [33]:
messages_client = "\n".join(m["content"] for m in historique if m["role"] == "user")

r = client.chat.completions.parse(
    model=MODELE,
    response_format=ProfilClient,                    # 👈 le format imposé
    messages=[{"role": "system", "content": "Remplis la fiche du client à partir de ses messages. N'invente rien."},
              {"role": "user", "content": messages_client}],
)
profil = r.choices[0].message.parsed                 # un vrai objet Python, déjà validé
print(profil.model_dump_json(indent=2))
print("\nPointure + 1 =", profil.pointure + 1, "(c'est bien un nombre)")

{
  "prenom": "Sofia",
  "pointure": 42,
  "couleurs_preferees": [
    "blanc"
  ],
  "budget_max": 130.0,
  "centres_interet": [
    "sport"
  ]
}

Pointure + 1 = 43 (c'est bien un nombre)


### 🔁 La même chose avec LangChain

`with_structured_output()` fait la même chose, quel que soit le fournisseur de LLM.

In [34]:
extracteur = llm.with_structured_output(ProfilClient)
profil_lc = extracteur.invoke(f"Remplis la fiche de ce client. N'invente rien.\n\n{messages_client}")
print(profil_lc)

prenom='Sofia' pointure=42 couleurs_preferees=['blanc'] budget_max=130.0 centres_interet=['sport']


### 🗓️ Une semaine plus tard...

Sofia revient. C'est une **nouvelle conversation** : l'historique est vide. Mais on glisse sa fiche dans le prompt système :

In [35]:
SYSTEME_PERSO = SYSTEME + f"\n\nFiche du client (mémoire long terme) : {profil.model_dump_json()}"

r = client.chat.completions.create(model=MODELE, messages=[
    {"role": "system", "content": SYSTEME_PERSO},
    {"role": "user", "content": "Re-bonjour ! Vous auriez des baskets pour moi ?"}])
print("💬 Léa :", r.choices[0].message.content)

💬 Léa : Bonjour Sofia, je suis ravie de vous retrouver ! Nous avons plusieurs baskets blanches en pointure 42, idéales pour le sport et à moins de 130€. Souhaitez-vous que je vous en sélectionne quelques-unes ?


🎉 Léa reconnaît Sofia et adapte son conseil (pointure, couleur, budget) **sans que Sofia ait rien répété**.

✅ **À retenir**
- La **mémoire long terme** se stocke **hors du LLM**, idéalement sous forme **structurée** (une fiche, une base de données).
- **Pydantic** décrit le format attendu. Le LLM le remplit, et Pydantic **garantit les types**.
- ⚠️ Une fiche client contient des **données personnelles** : en production, il faut respecter le RGPD (consentement, durée de conservation, droit à l'oubli).

---
## 🎓 Récapitulatif : les 4 mémoires d'un assistant

| Mémoire | Durée | Exemple chez MégaShop | Où l'a-t-on vue ? |
|---|---|---|---|
| **Historique** (court terme) | La conversation | « Je chausse du 42 », dit il y a 2 messages | Étape 5 |
| **Résumé** | La conversation, en compressé | « Sofia, 42, aime le blanc, 130 € max » | Étape 6 |
| **Profil client** (long terme) | Des mois | La fiche `ProfilClient` | Étape 7 |
| **Connaissances** | Permanente | Le catalogue, les CGV, le guide des tailles | Notebooks 2 et 3 |

| Concept | Python pur | Framework |
|---|---|---|
| Appeler un LLM | `client.chat.completions.create()` | `ChatOpenAI().invoke()` (LangChain) |
| Mémoire par conversation | une liste `historique` | `checkpointer` + `thread_id` (LangGraph) |
| Sortie structurée | `client.chat.completions.parse()` | `with_structured_output()` (LangChain) |

**Les 3 idées à garder**
1. Un LLM **prédit des tokens** : il ne « sait » rien de ta boutique et ne se souvient de rien.
2. La mémoire, **c'est ton code** qui la gère, et elle a un coût.
3. Pour une mémoire durable et fiable, on **structure** les données (Pydantic).

➡️ **Prochain notebook** : Léa sait parler, mais elle ne connaît pas le stock. Elle va apprendre à **utiliser des outils** (stock, livraison, retours) : c'est le passage de l'assistant à l'**agent**.